# K_07 – Technologievergleich

**Grid-Arbitrage** · Batteriespeicher-Arbitrage im Schweizer Strommarkt (Kür)

**Gruppe:** SC26_Gruppe_2 | **Verantwortlich:** Patrik Neunteufel | **Datum:** März 2026

---

*LFP, [NMC](../organisation/O_02_Glossar.ipynb#g-nmc), [NaS](../organisation/O_02_Glossar.ipynb#g-nas), [Redox-Flow](../organisation/O_02_Glossar.ipynb#g-redox-flow), Pumpspeicher: Technologieprofil und Segmenteignung.*


| [← K_06 – Dispatch-Optimierung](K_06_Dispatch_Optimierung.ipynb) | [↑ Übersicht ↑](../organisation/O_01_Project_Overview.ipynb) | [K_08 – Alternative Speicher →](K_08_Alternative_Speicher.ipynb) |
|:---|:---:|---:|

## Inhaltsverzeichnis<a id='toc_K_07'></a>

[Einleitung](#einleitung_K_07)  
[Initialisierung](#initialisierung_K_07)  
1 [Technologie-Steckbriefe](#technologie-steckbriefe_K_07)  
2 [Technologieprofil-Vergleich (Radar-Chart)](#technologieprofil-vergleich-radar-chart_K_07)  
3 [CAPEX-Lernkurven 2015–2024 + Projektion bis 2035](#capex-lernkurven-20152024-projektion-bis-2035_K_07)  
4 [Segmenteignung für Grid-Arbitrage CH](#segmenteignung-fuer-grid-arbitrage-ch_K_07)  
[Fazit](#fazit_K_07)  
[Abschluss](#abschluss_K_07)  


---
## Einleitung <a id='einleitung_K_07'></a>

[↑ Inhaltsverzeichnis](#toc_K_07)

Vergleich der relevanten Speicher-Technologien für Grid-Arbitrage:
Li-Ion (NMC und LFP), Redox-Flow, Natrium-Schwefel (NaS), Compressed-Air
Energy Storage (CAES) und Pumpspeicher (PHS).

- **Steckbriefe** — Kennzahlen je Technologie (CAPEX, RTE, Zyklen, Entladezeit)
- **Radar-Chart** — normierter Profil-Vergleich
- **CAPEX-Lernkurven** — historische Entwicklung 2015–2024 + Projektion bis 2035
- **Segmenteignung** — welche Technologie passt zu welchem Einsatzfall in der CH


## Initialisierung<a id='initialisierung_K_07'></a>

[↑ Inhaltsverzeichnis](#toc_K_07)

Bibliotheken laden, `../sync/config.json` lesen, Verzeichnispfade setzen.

**Imports und Versionen:**

In [ ]:
# ── lib/ aus Projekt-Root erreichbar machen + lib-Imports ───────────────────
import sys, os
def _find_project_root():
    cur = os.path.abspath('.')
    for _ in range(5):
        if os.path.isfile(os.path.join(cur, 'sync', 'config.json')):
            return cur
        cur = os.path.dirname(cur)
    return os.path.abspath('..')

_PROJECT_ROOT = _find_project_root()
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

from lib.io_ops    import load_transfer, final_check
from lib.plotting  import show_source

print(f'lib-Pfad aktiv: {_PROJECT_ROOT}/lib')


In [ ]:
# ── Bibliotheken ─────────────────────────────────────────────────────────────
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
warnings.filterwarnings('ignore')
from datetime import datetime

# Versionen anzeigen für Reproduzierbarkeit
print(f"Numpy        Version: {np.__version__}")
print(f"Pandas       Version: {pd.__version__}")
print(f"📅 Zuletzt ausgeführt am: {datetime.now().strftime('%d.%m.%Y um %H:%M:%S')}")

**Setup – Konfiguration & Verzeichnisstruktur:** Lädt `../sync/config.json` (SSOT), setzt Pfade.

In [ ]:
with open('../sync/config.json') as f:
    CFG = json.load(f)

SZ_AKTIV   = CFG['szenarien']['gleichzeitigkeit_aktiv']
FORCE_RELOAD = CFG.get('force_reload', {})  # konventionskonform gelesen
CHARTS_DIR = os.path.join('../output', 'charts', SZ_AKTIV)
os.makedirs(CHARTS_DIR, exist_ok=True)
DPI = CFG['visualisierung']['output_dpi']  # SSOT: ../sync/config.json

# ── Farben & Stil aus ../sync/config.json (SSOT) ─────────────────────────────────────
# Bestehende Variablen (Rückwärtskompatibilität)
_viz        = CFG.get('visualisierung', {}).get('farben', {})
BG_DARK     = _viz.get('bg_dark',    '#0d1117')
BG_PANEL    = _viz.get('bg_panel',   '#141414')
C_PRICE     = _viz.get('c_price',    '#FFA726')
C_LOAD      = _viz.get('c_load',     '#66BB6A')
C_CHARGE    = _viz.get('c_charge',   '#1565C0')
C_FEED      = _viz.get('c_feed',     '#B71C1C')
SEG_COLORS  = _viz.get('seg_colors', ['#42A5F5', '#66BB6A', '#FFA726', '#EF5350'])
C_PRIV, C_GEW, C_IND, C_UTIL = SEG_COLORS

# UI-Strukturfarben
C_ACHSE      = _viz.get('c_achse',      '#aaaaaa')  # Achsenbeschriftungen
C_TICK       = _viz.get('c_tick',       '#bbbbbb')  # Tick-Labels
C_SPINE      = _viz.get('c_spine',      '#333333')  # Achsenrahmen
C_LEGENDE_BG = _viz.get('c_legende_bg', '#111111')  # Legenden-Hintergrund
C_GITTER     = _viz.get('c_gitter',     '#cccccc')  # Gitterlinien

# Funktionale Extrafarben (nur laden was das NB braucht)
C_DISPATCH   = _viz.get('c_dispatch',   '#AB47BC')  # Dispatch-optimal
C_STACKING   = _viz.get('c_stacking',   '#5DCAA5')  # Revenue Stacking
C_SOLAR      = _viz.get('c_solar',      '#FDD835')  # Solar-Ertrag
C_GRENZWERT  = _viz.get('c_amber_dark', '#FF6F00')  # Grenzwert / Warnung
C_CYAN       = _viz.get('c_cyan',       '#26C6DA')  # Flusswasser / Alt. Speicher
C_GRUEN_DARK = _viz.get('c_gruen_dark', '#388E3C')  # Erneuerbare

# Stilkonstanten
_stil               = CFG.get('visualisierung', {}).get('stil', {})
LW                  = _stil.get('linienbreite_standard', 1.5)   # Standard-Linienbreite
LW_DUENN            = _stil.get('linienbreite_duenn',    0.8)   # dünne Linien
LW_DICK             = _stil.get('linienbreite_dick',     2.5)   # dicke Linien
ALPHA_FLAECHE       = _stil.get('alpha_flaeche',         0.12)  # dezente Füllung
ALPHA_FLAECHE_STARK = _stil.get('alpha_flaeche_stark',   0.35)  # Balken / Füllung
ALPHA_LEGENDE       = _stil.get('alpha_legende',         0.30)  # Legenden-BG
ALPHA_GEDAEMPFT     = _stil.get('alpha_linie_gedaempft', 0.55)  # Nebenlinien
FS_TITEL            = _stil.get('schriftgroesse_titel',   13)   # Chart-Titel
FS_ACHSE            = _stil.get('schriftgroesse_achse',   10)   # Achsenbeschr.
FS_TICK             = _stil.get('schriftgroesse_tick',     9)   # Ticks
FS_LEGENDE          = _stil.get('schriftgroesse_legende',  8)   # Legende
FS_KLEIN            = _stil.get('schriftgroesse_klein',    7)   # Annotationen

# matplotlib rcParams — nur stabile, versionsunabhängige Keys (matplotlib >= 3.5)
# axes.titlecolor (3.8+) und axes.grid (stört Karten) bewusst NICHT gesetzt
import matplotlib as mpl
mpl.rcParams.update({
    'figure.facecolor':  BG_DARK,
    'axes.facecolor':    BG_PANEL,
    'axes.edgecolor':    C_SPINE,
    'axes.labelcolor':   C_ACHSE,
    'axes.labelsize':    FS_ACHSE,
    'axes.titlesize':    FS_TITEL,
    'xtick.color':       C_TICK,
    'ytick.color':       C_TICK,
    'xtick.labelsize':   FS_TICK,
    'ytick.labelsize':   FS_TICK,
    'text.color':        'white',
    'lines.linewidth':   LW,
    'legend.facecolor':  C_LEGENDE_BG,
    'legend.framealpha': ALPHA_LEGENDE,
    'legend.fontsize':   FS_LEGENDE,
    'legend.edgecolor':  C_SPINE,
})
print('Farben & Stil geladen.')

print(f"Setup OK | Szenario={SZ_AKTIV} | Charts → {CHARTS_DIR}")

**⚙ Markdown-Prüfwerte (config-abhängig) und 📊 Markdown-Prüfwerte (transfer-abhängig)**:  
Werte mit ⚙ kommen aus `../sync/config.json`, Werte mit 📊 aus `../sync/transfer.json` (NB03-Output).  
Bei jeder Änderung dieser Quellen → Output mit ⚙/📊-Stellen im Markdown abgleichen.


In [ ]:
# ── ⚙ Markdown-Prüfwerte (config-abhängig) ─────────────────────────────────
print('=== ⚙ MARKDOWN-PRÜFWERTE (config-abhängig) ===')
_pflicht = CFG['pflicht']['simulation']
_wirt    = CFG['pflicht']['wirtschaftlichkeit']
_markt   = CFG['kuer']['markt']
print(f'  Round-Trip-Effizienz       = {_pflicht["efficiency_roundtrip"]*100:.0f} %  (LFP-Annahme NB02/04)')
print(f'  Lebensdauer (NB02)         = {_wirt["lifetime_j"]} Jahre  (LFP konservativ)')
print(f'  CAPEX-Trigger Privat       = {_markt["capex_ziel_privat_eur_kwh"]} EUR/kWh installiert')
print(f'  CAPEX-Lernrate-Annahme     = {_markt["capex_lernrate_pct_pro_jahr"]} %/Jahr (Schätzung K_03)')


**🔎 Quellcode der importierten lib-Funktion**

`load_transfer` aus `lib.io_ops` — liest `../sync/transfer.json`.


In [ ]:
show_source(load_transfer)


In [ ]:
# ── 📊 Markdown-Prüfwerte (transfer-abhängig) ──────────────────────────────
TF      = load_transfer()
_tf_dz  = TF.get('datenzeitraum', {})
print('=== 📊 MARKDOWN-PRÜFWERTE (transfer-abhängig) ===')
print(f'  Datenzeitraum         = {_tf_dz.get("start_date","?")} bis {_tf_dz.get("end_date","?")} ({_tf_dz.get("n_years","?")}J)')
print(f'  (K_07 nutzt keine NB03-Simulationswerte direkt — Tech-Vergleich')
print(f'   ist literaturbasiert; OWID-CAPEX wird aus dem Web nachgeladen.)')


---
## 1. Technologie-Steckbriefe <a id='technologie-steckbriefe_K_07'></a>

[↑ Inhaltsverzeichnis](#toc_K_07)

Kennwerte 2024/2025 — Quellen: Bloomberg NEF, IRENA, Fraunhofer ISE.  
Alle Werte: System-[CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) (Zellen + Wechselrichter + Installation), ohne Gebäude/Fundament.


In [ ]:
# ── Technologievergleich-Tabelle ─────────────────────────────────────────────
tech_data = {
    'Technologie'         : ['LFP (Li-Ion)', 'NMC (Li-Ion)', 'Redox-Flow (VRF)', 'NaS'],
    'CAPEX [EUR/kWh]'     : ['150–300',       '200–400',       '400–800',          '300–500'],
    'Lebensdauer [J]'     : ['15–20',          '10–15',          '20–25',            '15–20'],
    'Zyklen'              : ['3 000–6 000',    '1 500–3 000',    '> 20 000',         '4 500+'],
    'RTE [%]'             : ['92–95',          '90–93',          '70–80',            '75–85'],
    'Selbstentl./Tag'     : ['2–3 %',          '2–5 %',          '< 1 %',            '~15 %'],
    'Betriebstemperatur'  : ['-20…+60 °C',     '-20…+60 °C',     '+10…+40 °C',       '+300…+350 °C'],
    'Skalierbarkeit'      : ['Modular, gut',   'Modular, gut',   'Sehr hoch',        'Mittel'],
    'Brandsicherheit'     : ['Hoch (LFP)',      'Mittel (NMC)',   'Unkritisch',       'Betriebswärme'],
    'Typisches Segment'   : ['Privat/Gewerbe', 'EV / mobil',     'Utility/Netz',     'Utility/Wärme'],
}
df_tech = pd.DataFrame(tech_data).set_index('Technologie')
display(df_tech.T)



**Projektrelevanz:** NB04 verwendet LFP als Referenztechnologie (RTE = 92 %, [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) 150–400 EUR/kWh je Segment).
Diese Wahl ist korrekt: LFP dominiert den CH-Stationärspeichermarkt 2023–2026.


---
## 2. Technologieprofil-Vergleich (Radar-Chart) <a id='technologieprofil-vergleich-radar-chart_K_07'></a>

[↑ Inhaltsverzeichnis](#toc_K_07)

Normierte Scores (1–10): höher = besser für [Grid-Arbitrage](../organisation/O_02_Glossar.ipynb#g-grid-arbitrage).  
Scoring-Kriterien: [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) (invertiert), Lebensdauer, Zyklenanzahl, Wirkungsgrad, Sicherheit, Skalierbarkeit.


In [ ]:
# ── Radar-Chart ──────────────────────────────────────────────────────────────
scores = {
    'LFP'        : [8.5, 7.0, 9.5, 9.5, 9.0, 8.0],
    'NMC'        : [7.0, 5.5, 6.0, 8.5, 5.0, 7.5],
    'Redox-Flow' : [4.0, 9.0, 10.0, 6.0, 10.0, 10.0],
    'NaS'        : [6.0, 7.0, 8.0, 7.0, 4.0, 5.0],
}
labels = ['CAPEX\n(inv.)', 'Lebens-\ndauer', 'Zyklen', 'Wirkungs-\ngrad', 'Sicherheit', 'Skalierung']
N = len(labels)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]
colors_t = [C_PRIV,C_PRICE,C_LOAD,C_UTIL]

fig, ax = plt.subplots(figsize=(8, 7), subplot_kw=dict(polar=True))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
ax.tick_params(colors=C_TICK); ax.grid(color=C_SPINE, lw=0.7)
ax.spines['polar'].set_color('#444')

for (name, sc), col in zip(scores.items(), colors_t):
    vals = sc + [sc[0]]
    ax.plot(angles, vals, color=col, lw=2.2, label=name)
    ax.fill(angles, vals, color=col, alpha=0.10)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, color=C_GITTER, fontsize=FS_TICK)
ax.set_ylim(0, 10)
ax.set_yticks([2,4,6,8,10])
ax.set_yticklabels(['2','4','6','8','10'], color='#555', fontsize=FS_KLEIN)
ax.set_title('Technologieprofil — stationäre Grid-Arbitrage', color='white', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.12),
          framealpha=ALPHA_LEGENDE, facecolor=C_LEGENDE_BG, labelcolor='white', fontsize=FS_TICK)

p = os.path.join(CHARTS_DIR, 'kuer_k07_radar.png')
plt.savefig(p, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f"Gespeichert: {p}")


---
## 3. CAPEX-Lernkurven 2015–2024 + Projektion bis 2035 <a id='capex-lernkurven-20152024-projektion-bis-2035_K_07'></a>

[↑ Inhaltsverzeichnis](#toc_K_07)

### Datenquellen

| Quelle | Inhalt | Zugang |
|--------|--------|--------|
| **Our World in Data** (BNEF/Ziegler & Trancik) | Historische Zellpreise 1991–2024, USD/kWh | Freier CSV-Download |
| **NREL ATB 2024** | Systemkosten + Projektionsszenarien 2022–2050 | Freier CSV-Download (AWS S3) |
| **BloombergNEF Battery Price Survey** | Umfassendste Datenbasis: Zell- und Systemkosten, Chemie-Aufschlüsslung, Regionen | **Bezahlschranke** |

> **Hinweis zu BNEF:** BloombergNEF ist die Referenzquelle der Branche für Batteriepreis-Daten
> (auch NREL ATB und OWID beruhen partiell darauf). Für geschäftliche Entscheidungen
> — z.B. konkrete Investitionsrechnung oder Due-Diligence — wäre ein BNEF-Abonnement
> in Erwägung zu ziehen.

**Methodik:** Exponentielle Regression auf historischen Daten (Proxy für Wright’s Law);
NREL ATB Szenarien als zusätzliche Referenzkurven.


In [ ]:
# ── CAPEX-Lernkurven: OWID + NREL ATB CSV + Regression ───────────────────────
import requests, io  # Re-Import mit lokalem Alias (NREL ATB CSV Download)
import scipy
from scipy.optimize import curve_fit
# ════════════════════════════════════════════════════════════════════════════
# DATENQUELLE 1: Our World in Data — historische Li-Ion Zellpreise
# Rupert Way (2026) auf Basis BNEF, Ziegler & Trancik (2021), Avicenne Energy
# Direkt-CSV: https://ourworldindata.org/grapher/price-of-lithium-ion-battery-cells.csv
# ════════════════════════════════════════════════════════════════════════════
OWID_URL = 'https://ourworldindata.org/grapher/price-of-lithium-ion-battery-cells.csv'

df_hist = None
try:
    resp = requests.get(OWID_URL, timeout=15)
    resp.raise_for_status()
    _df  = pd.read_csv(io.StringIO(resp.text))
    # Spaltenname für Preis finden (variiert je nach OWID-Version)
    col_price = next((c for c in _df.columns
                      if any(k in c.lower() for k in ['price','lithium','cost','value'])), _df.columns[-1])
    col_year  = next((c for c in _df.columns if 'year' in c.lower() or c == 'Year'), 'Year')
    df_hist = (_df.groupby(col_year)[col_price].mean()
                  .reset_index().rename(columns={col_year: 'Year', col_price: 'cell_usd_kwh'}))
    # Systemkosten: Zellkosten × 2.5 (BOS + WR + Installation) × 0.92 (USD→EUR)
    df_hist['system_eur_kwh'] = df_hist['cell_usd_kwh'] * 2.5 * 0.92
    df_hist = df_hist[df_hist['Year'] >= 2015].sort_values('Year').dropna()
    yr_last  = int(df_hist['Year'].max())
    val_last = df_hist.loc[df_hist['Year'] == yr_last, 'system_eur_kwh'].values[0]
    print(f"OWID geladen: {len(df_hist)} Jahre ({int(df_hist['Year'].min())}–{yr_last})")
    print(f"  System-CAPEX {yr_last}: ~{val_last:.0f} EUR/kWh")
    print(f"  Quelle: {OWID_URL}")
except Exception as e:
    print(f"⚠️  OWID-Download fehlgeschlagen: {e}")
    print("   Fallback: eingebettete BNEF-Referenzwerte (Systemkosten EUR/kWh)")
    df_hist = pd.DataFrame({'Year': [2015,2016,2017,2018,2019,2020,2021,2022,2023,2024],
                            'system_eur_kwh': [530,445,380,310,255,215,190,178,160,145]})

years_h = df_hist['Year'].values.astype(float)
capex_h = df_hist['system_eur_kwh'].values.astype(float)

# ════════════════════════════════════════════════════════════════════════════
# DATENQUELLE 2: NREL ATB 2024 — Projektionsszenarien
# CSV aus OEDI Data Lake (AWS S3, öffentlich, kein Auth)
# https://oedi-data-lake.s3.amazonaws.com/ATB/electricity/csv/2024/v3.0.0/ATBe.csv
# ════════════════════════════════════════════════════════════════════════════
ATB_URL = ('https://oedi-data-lake.s3.amazonaws.com'
           '/ATB/electricity/csv/2024/v3.0.0/ATBe.csv')

# ATB CAPEX-Werte für Utility-Scale Battery (4h), $/kW → $/kWh = ÷4 → EUR/kWh × 0.92
# Struktur: technology | techdetail | scenario | year | parameter | value | units
ATB_SCENARIOS = None
try:
    print(f"\nLade NREL ATB 2024: {ATB_URL}")
    resp2 = requests.get(ATB_URL, timeout=30)
    resp2.raise_for_status()
    df_atb = pd.read_csv(io.StringIO(resp2.text))
    print(f"  ATB geladen: {df_atb.shape[0]:,} Zeilen | Spalten: {list(df_atb.columns)}")

    # Spalten normalisieren
    df_atb.columns = [c.lower().strip() for c in df_atb.columns]

    # Battery Storage filtern — präzise Filter um 19'314 → ein paar Hundert Zeilen
    # WICHTIG: Pattern (1) core_metric_parameter == 'CAPEX' (nicht 'Construction Financing CAPEX' etc.)
    #          (2) techdetail == '4Hr Battery Storage' (nicht 2Hr/6Hr/8Hr/10Hr)
    #          (3) crpyears == 20 (Standard-Lebensdauer)
    #          (4) core_metric_case == 'Market' (nicht 'R&D')
    #          (5) Projektionsjahr aus 'core_metric_variable' (nicht 'atb_year')
    df_bat = df_atb[
        df_atb['technology'].str.lower().str.contains('utility-scale battery', na=False) &
        (df_atb['core_metric_parameter'] == 'CAPEX') &
        (df_atb['techdetail'] == '4Hr Battery Storage') &
        (df_atb['crpyears'] == 20) &
        (df_atb['core_metric_case'].str.lower() == 'market') &
        (df_atb['tax_credit_case'].str.lower() == 'no itc')
    ].copy()
    print(f"  Battery+CAPEX Zeilen (gefiltert): {len(df_bat)}")
    print(f"  Szenarien: {sorted(df_bat['scenario'].unique())}")

    # ATB liefert Utility-Scale Battery in USD/kW (Power-Capacity)
    # 4h-System → USD/kWh = USD/kW / 4 ; EUR/kWh = USD/kWh × 0.92
    df_bat['capex_eur_kwh'] = pd.to_numeric(df_bat['value'], errors='coerce') / 4 * 0.92

    # Projektionsjahr = core_metric_variable (Zeitachse 2022–2050)
    ATB_SCENARIOS = {}
    for scen in ['Conservative', 'Moderate', 'Advanced']:
        df_s = df_bat[df_bat['scenario'] == scen]
        if not df_s.empty:
            ts = (df_s.groupby('core_metric_variable')['capex_eur_kwh'].mean()
                      .reset_index().sort_values('core_metric_variable')
                      .rename(columns={'core_metric_variable': 'year'}))
            ATB_SCENARIOS[scen] = ts
            yr0, c0 = int(ts['year'].iloc[0]), ts['capex_eur_kwh'].iloc[0]
            yr1, c1 = int(ts['year'].iloc[-1]), ts['capex_eur_kwh'].iloc[-1]
            print(f"  Szenario {scen:12s}: {yr0}={c0:.0f} EUR/kWh → {yr1}={c1:.0f} EUR/kWh")
    print(f"  Quelle: {ATB_URL}")
except Exception as e:
    print(f"⚠️  NREL ATB Download fehlgeschlagen: {e}")
    print("   Fallback: NREL ATB Referenzwerte (embedded, 2022 Basisjahr, 4h System)")
    # Referenzwerte aus ATB 2024 Dokumentation (USD/kW ÷ 4 × 0.92)
    base_2022 = 185.0   # EUR/kWh (Utility 4h, 2022)
    proj_yrs  = list(range(2022, 2051))
    ATB_SCENARIOS = {
        'Conservative': pd.DataFrame({'year': proj_yrs,
            'capex_eur_kwh': [base_2022 * (1-0.014)**i for i in range(len(proj_yrs))]}),
        'Moderate':     pd.DataFrame({'year': proj_yrs,
            'capex_eur_kwh': [base_2022 * (1-0.029)**i for i in range(len(proj_yrs))]}),
        'Advanced':     pd.DataFrame({'year': proj_yrs,
            'capex_eur_kwh': [base_2022 * (1-0.040)**i for i in range(len(proj_yrs))]}),
    }
    print("   Raten: Conservative -1.4%/J | Moderate -2.9%/J | Advanced -4.0%/J")

# ════════════════════════════════════════════════════════════════════════════
# REGRESSION: Exponentielle Kurve auf historische OWID-Daten
# Proxy für Wright's Law auf der Zeitachse
# ════════════════════════════════════════════════════════════════════════════
def exp_decay(t, a, b):
    return a * np.exp(-b * t)

t_h = years_h - years_h[0]
try:
    popt, _ = curve_fit(exp_decay, t_h, capex_h, p0=[capex_h[0], 0.08], maxfev=5000)
    a_fit, b_fit = popt
    rate_pct = (1 - np.exp(-b_fit)) * 100
    print(f"\nRegression (Wright's Law Proxy):")
    print(f"  CAPEX(t) = {a_fit:.0f} × e^(−{b_fit:.4f}·t)  |  ~{rate_pct:.1f}%/Jahr historisch")
except Exception as e:
    print(f"Regression fehlgeschlagen: {e}"); b_fit = np.log(1/0.90); a_fit = capex_h[-1]

# Regressionskurve bis 2035 extrapolieren
base_yr    = int(years_h[-1])
base_capex = float(capex_h[-1])
proj_yrs_r = np.arange(base_yr, 2036)
proj_reg   = base_capex * np.exp(-b_fit * (proj_yrs_r - base_yr))

# ════════════════════════════════════════════════════════════════════════════
# CHART
# ════════════════════════════════════════════════════════════════════════════
SCEN_STYLE = {
    'Conservative': (C_PRICE, '--'),
    'Moderate':     (C_LOAD, '--'),
    'Advanced':     (C_UTIL, ':'),
}

fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
ax.tick_params(colors=C_TICK)
for sp in ax.spines.values(): sp.set_edgecolor(C_SPINE)

# Historische Punkte (OWID)
ax.scatter(years_h, capex_h, color='white', s=45, zorder=6,
           label=f'Historisch — OWID/BNEF (Systemkosten)', alpha=0.9)
ax.plot(years_h, capex_h, color='white', lw=1.3, alpha=0.35)

# Regressionskurve
ax.plot(proj_yrs_r, proj_reg, color=C_PRIV, lw=2.2, ls='-',
        label=f'Regression histor. Daten (~{rate_pct:.0f}%/J)', alpha=0.9)

# NREL ATB Szenarien
if ATB_SCENARIOS:
    for scen, df_s in ATB_SCENARIOS.items():
        col, ls = SCEN_STYLE.get(scen, ('#888888', ':'))
        mask = df_s['year'] >= base_yr
        if mask.any():
            ax.plot(df_s.loc[mask, 'year'], df_s.loc[mask, 'capex_eur_kwh'],
                    color=col, lw=2, ls=ls, label=f'NREL ATB {scen}', alpha=0.9)

# Referenzlinien
ax.axvline(base_yr, color='white', lw=0.7, ls=':', alpha=0.3)
ax.axhline(250, color=C_UTIL, lw=1.1, ls=':', alpha=0.6,
           label='Trigger-CAPEX 250 EUR/kWh (NB04)')
ax.axhline(150, color=C_DISPATCH, lw=1.0, ls=':', alpha=0.5,
           label='Langfrist-Ziel 150 EUR/kWh')
ax.axvspan(base_yr, 2035, alpha=0.04, color='white')
ax.text(base_yr + 0.3, max(capex_h) * 1.04, 'Projektion →', color='#666', fontsize=FS_LEGENDE)

ax.set_xlabel('Jahr', color=C_ACHSE, fontsize=11)
ax.set_ylabel('System-CAPEX [EUR/kWh]\n'
              '(Zellen + WR + Installation, 4h-System, 2024-Preise)', color=C_ACHSE, fontsize=FS_ACHSE)
ax.set_title('Li-Ion Batteriespeicher — CAPEX-Entwicklung & Projektion\n'
             'Historisch: OWID/BNEF  |  Szenarien: NREL ATB 2024 (CSV)',
             color='white', fontweight='bold')
ax.legend(fontsize=8.5, framealpha=ALPHA_LEGENDE, facecolor=C_LEGENDE_BG, labelcolor='white',
          loc='upper right')
ax.grid(True, alpha=ALPHA_FLAECHE)
ax.set_xlim(2014.5, 2035.5)

p = os.path.join(CHARTS_DIR, 'kuer_k07_capex_lernkurven.png')
plt.savefig(p, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f'\nChart gespeichert: {p}')

**Befund:** Die historische Regression zeigt einen Rückgang von ~8–12 %/Jahr.
Das NREL ATB Moderate-Szenario (−2.9 %/Jahr) entspricht dem langfristigen Konsens;
das Advanced-Szenario (−4.0 %/Jahr) spiegelt aktuelle Dynamiken wider —
nach einem ~40 %-Rückgang 2024 liegen globale Projektkosten bereits bei ~125 USD/kWh
(Ember Energy, Januar 2026).

**Wichtig — Marktsegment-Unterscheidung:** Alle Projekt-Segmente (Privat 10 kWh
bis Utility 10 MWh) verwenden **dieselbe LFP-Technologie** und **dieselbe Konverter-Klasse**
(0.5C–1C, modular skaliert). Die CAPEX-Differenz kommt **nicht** aus Technologie-Wechsel,
sondern aus **Skalenökonomie**:

| Segment | CAPEX (NB04)📊 | Begründung |
|---|---|---|
| **Privat** 10 kWh | ~400 EUR/kWh | Hoher Anteil fixer BOS-Kosten (Wechselrichter, Engineering, Installation) + Vertriebsmarge pro Anlage |
| **Gewerbe** 100 kWh | ~300 EUR/kWh | BOS-Kosten verteilen sich auf 10× mehr Kapazität |
| **Industrie** 1 MWh | ~220 EUR/kWh | Großserien-Beschaffung, weniger Marge |
| **Utility** 10 MWh | ~180 EUR/kWh | Container-Standardlösungen, Direktverkauf, Volumenrabatte bei Zellen (BNEF) |

OWID/NREL ATB zeigen **Utility-Scale**-CAPEX (~180 EUR/kWh 2024) — diese
Werte sind nicht direkt mit dem **Privat-Trigger 250 EUR/kWh⚙** aus K_03 vergleichbar.
Beide Segmente folgen aber **derselben Lernkurve (~10 %/Jahr⚙)**, da die Zellen identisch sind.

**Befund Utility-Segment:** Der Trigger-[CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) von 250 EUR/kWh wird je nach Szenario
zwischen 2025 (Advanced) und 2028 (Conservative) unterschritten — Utility-Speicher
sind also **bereits heute** wirtschaftlich nahe am Break-Even, während der Privat-Trigger
~2029⚙ erreicht wird (siehe K_03 Sektion 3).

> **Datenquellen:**
> - [OWID — Li-Ion Battery Prices](https://ourworldindata.org/grapher/price-of-lithium-ion-battery-cells) (BNEF/Ziegler & Trancik, freier Download)
> - [NREL ATB 2024 — Utility-Scale Battery](https://atb.nrel.gov/electricity/2024/utility-scale_battery_storage) (CSV via AWS S3, freier Download)
> - **BloombergNEF Battery Price Survey** — umfassendste Branchendatenbank (Bezahlschranke; für geschäftliche Due-Diligence empfehlenswert)


---
## 4. Segmenteignung für Grid-Arbitrage CH <a id='segmenteignung-fuer-grid-arbitrage-ch_K_07'></a>

[↑ Inhaltsverzeichnis](#toc_K_07)

Eignungsscores 1–5 (5 = ideal) für die vier Projektsegmente.


In [ ]:
# ── Eignungsmatrix (Heatmap) ──────────────────────────────────────────────────
techs    = ['LFP', 'NMC', 'Redox-Flow', 'NaS']
segs     = ['Privat\n10 kWh', 'Gewerbe\n100 kWh', 'Industrie\n1 MWh', 'Utility\n10 MWh']
eignung  = np.array([
    [5, 4, 3, 2],   # LFP
    [3, 3, 2, 1],   # NMC
    [1, 2, 4, 5],   # Redox-Flow
    [1, 2, 3, 4],   # NaS
])
sterne = ['','★','★★','★★★','★★★★','★★★★★']

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
cmap = mcolors.LinearSegmentedColormap.from_list('', ['#1a1a1a','#1a3d2b',C_STACKING])
im = ax.imshow(eignung, cmap=cmap, vmin=1, vmax=5, aspect='auto')
ax.set_xticks(range(len(segs)));  ax.set_xticklabels(segs, color=C_GITTER, fontsize=FS_ACHSE)
ax.set_yticks(range(len(techs))); ax.set_yticklabels(techs, color=C_GITTER, fontsize=FS_ACHSE)
for i in range(len(techs)):
    for j in range(len(segs)):
        ax.text(j, i, sterne[eignung[i,j]], ha='center', va='center', color='white', fontsize=FS_TITEL)
ax.set_title('Technologie-Segmenteignung — Grid-Arbitrage CH', color='white', fontweight='bold')
cb = plt.colorbar(im, ax=ax, label='Eignung (1=gering, 5=ideal)', shrink=0.85)
cb.ax.tick_params(colors=C_ACHSE); cb.ax.yaxis.label.set_color(C_ACHSE)
p = os.path.join(CHARTS_DIR, 'kuer_k07_eignungsmatrix.png')
plt.savefig(p, dpi=DPI, bbox_inches='tight', facecolor=BG_DARK)
plt.show(); plt.close()
print(f"Gespeichert: {p}")


**Kernaussagen:**
- **LFP** ist die dominierende Technologie für Privat und Gewerbe — beste Sicherheit, tiefster [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex)-Trend, marktreif.
- **NMC** verliert im stationären Bereich gegenüber LFP (Brandrisiko, kein CAPEX-Vorteil). Bleibt relevant für EV.
- **Redox-Flow** ist die strategisch interessanteste Alternative für Utility ≥ 10 MWh:
  nahezu unbegrenzte Zyklenlebensdauer, entkoppelbare Leistung/Energie, kaum [Degradation](../organisation/O_02_Glossar.ipynb#g-degradation).
- **NaS** benötigt 300–350 °C Betriebstemperatur — für dezentrale CH-Anlagen ungeeignet.


---
## Fazit <a id='fazit_K_07'></a>

[↑ Inhaltsverzeichnis](#toc_K_07)

**LFP ist die Referenztechnologie für alle Projektsegmente** — die in NB04 verwendeten
Parameter (RTE 92 %, [CAPEX](../organisation/O_02_Glossar.ipynb#g-capex) 150–400 EUR/kWh) sind korrekt und zeitgemäss.

Für Utility-Grossspeicher > 10 MWh ist **Redox-Flow** eine strategisch valide Alternative
mit deutlich längerer Lebensdauer (> 20 Jahre, > 20 000 Zyklen) und ohne Brandrisiko.
Der höhere CAPEX (400–800 EUR/kWh) amortisiert sich bei hohen Zyklenanzahlen.

> Für nicht-elektrochemische Alternativen (Pumpspeicher, [CAES](../organisation/O_02_Glossar.ipynb#g-caes), Power-to-X)  
> → [K_08 Alternative Speicher](K_08_Alternative_Speicher.ipynb)


---
## Abschluss <a id='abschluss_K_07'></a>

[↑ Inhaltsverzeichnis](#toc_K_07)

Erzeugte Charts (Radar, Lernkurven, Segmenteignung) auf Existenz prüfen —
werden von K_00 (Business Strategy) referenziert.

**🔎 Quellcode der importierten lib-Funktion**

Die Funktion `final_check` wird aus `lib/io_ops.py` importiert und
in der folgenden Zelle verwendet. Aufklappbar ist der Quellcode einsehbar


In [ ]:
show_source(final_check)

In [ ]:
# ── Abschlusskontrolle K_07 ─────────────────────────────────────────────────
final_check(
    'K_07',
    files=[
        (os.path.join(CHARTS_DIR, 'kuer_k07_radar.png'),           'Radar-Chart Technologieprofil', 30_000),
        (os.path.join(CHARTS_DIR, 'kuer_k07_capex_lernkurven.png'),'CAPEX-Lernkurven OWID + NREL ATB', 50_000),
        (os.path.join(CHARTS_DIR, 'kuer_k07_eignungsmatrix.png'),  'Eignungsmatrix Tech × Segment', 30_000),
    ],
    weiter_msg='K_08 Alternative Speicher',
)


---

| [← K_06 – Dispatch-Optimierung](K_06_Dispatch_Optimierung.ipynb) | [↑ Übersicht](../organisation/O_01_Project_Overview.ipynb) | [K_08 – Alternative Speicher →](K_08_Alternative_Speicher.ipynb) |
|:---|:---:|---:|